In [1]:
import os

from dotenv import load_dotenv
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import FireCrawlLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

In [2]:
load_dotenv()

True

In [3]:
current_dir = os.getcwd()
db_dir = os.path.join(current_dir, "db")
persistent_directory = os.path.join(db_dir, "chroma_db_firecrawl")
print(current_dir)

/home/kazi/Works/Projects/lang-chain/rag


In [4]:
def create_vector_store():
    """Crawl the website, split the content, create embeddings, and persist the vector store."""
    # Define the Firecrawl API key
    api_key = os.getenv("FIRECRAWL_API_KEY")
    if not api_key:
        raise ValueError("FIRECRAWL_API_KEY environment variable not set")
    loader = FireCrawlLoader(
        api_key=api_key, url="https://apple.com", mode="scrape")
    docs = loader.load()
    
    # Convert metadata values to strings if they are lists
    for doc in docs:
        for key, value in doc.metadata.items():
            if isinstance(value, list):
                doc.metadata[key] = ", ".join(map(str, value))

    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    split_docs = text_splitter.split_documents(docs)

    print(f"Number of document chunks: {len(split_docs)}")
    print(f"Sample chunk:\n{split_docs[0].page_content}\n")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    print(f"\n--- Creating vector store in {persistent_directory} ---")
    db = Chroma.from_documents(
        split_docs, embeddings, persist_directory=persistent_directory
    )
    print(f"--- Finished creating vector store in {persistent_directory} ---")

In [5]:
if not os.path.exists(persistent_directory):
    create_vector_store()
else:
    print(
        f"Vector store {persistent_directory} already exists. No need to initialize.")

Created a chunk of size 2427, which is longer than the specified 1000


Number of document chunks: 16
Sample chunk:
# Apple

- [Apple](https://www.apple.com/)
- - [Store](https://www.apple.com/us/shop/goto/store)


- [Mac](https://www.apple.com/mac/)

- [iPad](https://www.apple.com/ipad/)

- [iPhone](https://www.apple.com/iphone/)

- [Watch](https://www.apple.com/watch/)

- [Vision](https://www.apple.com/apple-vision-pro/)

- [AirPods](https://www.apple.com/airpods/)

- [TV & Home](https://www.apple.com/tv-home/)

- [Entertainment](https://www.apple.com/entertainment/)

- [Accessories](https://www.apple.com/us/shop/goto/buy_accessories)

- [Support](https://support.apple.com/?cid=gn-ols-home-hp-tab)

- [Search apple.com](https://www.apple.com/us/search)

- [Shopping Bag](https://www.apple.com/us/shop/goto/bag) 0+


## iPhone

Meet the iPhone 16 family.

[Learn more](https://www.apple.com/iphone/) [Shop iPhone](https://www.apple.com/us/shop/goto/buy_iphone)

Built for Apple Intelligence.

## Apple Watch Series 10

Thinstant classic.


--- Creating vector st

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(persist_directory=persistent_directory,
            embedding_function=embeddings)

In [7]:
def query_vector_store(query):
    retriever = db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3},
    )

    # Retrieve relevant documents based on the query
    relevant_docs = retriever.invoke(query)

    # Display the relevant results with metadata
    for i, doc in enumerate(relevant_docs, 1):
        print(f"Document {i}:\n{doc.page_content}\n")
        if doc.metadata:
            print(f"Source: {doc.metadata.get('source', 'Unknown')}\n")

In [8]:
query = "Apple Intelligence?"
query_vector_store(query)

Document 1:
- [Accessibility](https://www.apple.com/accessibility/)
- [Education](https://www.apple.com/education-initiative/)
- [Environment](https://www.apple.com/environment/)
- [Inclusion and Diversity](https://www.apple.com/diversity/)
- [Privacy](https://www.apple.com/privacy/)
- [Racial Equity and Justice](https://www.apple.com/racial-equity-justice-initiative/)
- [Supply Chain](https://www.apple.com/supply-chain/)

### About AppleAbout Apple

- [Newsroom](https://www.apple.com/newsroom/)
- [Apple Leadership](https://www.apple.com/leadership/)
- [Career Opportunities](https://www.apple.com/careers/us/)
- [Investors](https://investor.apple.com/)
- [Ethics & Compliance](https://www.apple.com/compliance/)
- [Events](https://www.apple.com/apple-events/)
- [Contact Apple](https://www.apple.com/contact/)

More ways to shop: [Find an Apple Store](https://www.apple.com/retail/) or [other retailer](https://locate.apple.com/) near you. Or call [1-800-MY-APPLE](tel:1-800-692-7753) (1-800-6